SÍ, LO DE ABAJO LO HA HECHO DEEPSEEK. 
SHHHHHH

In [ ]:
# %% [markdown]
# # 📊 Análisis de KPIs - FinPlus Analytics Challenge
# 
# **Objetivo:** Calcular los KPIs requeridos y generar insights accionables
# 
# **Entrada:** Datos limpios (después del ETL)
# **Salida:** KPIs, visualizaciones, insights para el dashboard
# 

# %% [code]
# ============================================================================
# CELDA 1: CONFIGURACIÓN INICIAL
# ============================================================================
import sys
import os
sys.path.append('../src')  # Para importar módulos propios

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Inicializar Spark
spark = SparkSession.builder \
    .appName("FinPlus_KPIs") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("✅ Spark inicializado")

# %% [code]
# ============================================================================
# CELDA 2: CARGA DE DATOS LIMPIOS
# ============================================================================
print("📂 Cargando datos limpios...")

# Cargar desde la capa procesada
try:
    # Intentar cargar Parquet (si ya procesaste los datos)
    df_clients = spark.read.parquet("../data/processed/clients_clean.parquet")
    df_behaviour = spark.read.parquet("../data/processed/behaviour_clean.parquet")
    print("✅ Datos cargados desde Parquet")
except:
    # Si no existen, cargar CSV y limpiar en el momento
    print("⚠️ No se encontraron datos procesados. Cargando CSV original...")
    df_clients = spark.read.csv("../data/CLIENTS.csv", header=True, inferSchema=True)
    df_behaviour = spark.read.csv("../data/BEHAVIOURAL.csv", header=True, inferSchema=True)
    
    # Limpieza básica
    df_clients = df_clients.dropDuplicates(["CLIENT_ID"])
    df_behaviour = df_behaviour.dropDuplicates()

print(f"📊 Clientes: {df_clients.count():,} filas, {len(df_clients.columns)} columnas")
print(f"📊 Comportamiento: {df_behaviour.count():,} filas, {len(df_behaviour.columns)} columnas")

# %% [code]
# ============================================================================
# CELDA 3: PREPARACIÓN DE DATOS PARA KPIs
# ============================================================================
print("🔧 Preparando datos para cálculo de KPIs...")

# Convertir fechas si es necesario
if "DATE" in df_behaviour.columns:
    df_behaviour = df_behaviour.withColumn("DATE", to_date(col("DATE"), "dd/MM/yyyy"))

# Crear columnas de tiempo
df_behaviour = df_behaviour.withColumn("YEAR", year(col("DATE"))) \
                           .withColumn("MONTH", month(col("DATE"))) \
                           .withColumn("YEAR_MONTH", date_format(col("DATE"), "yyyy-MM"))

# Calcular recencia (días desde última actividad)
ultima_fecha = df_behaviour.agg(max("DATE")).collect()[0][0]
df_behaviour = df_behaviour.withColumn("DAYS_SINCE_ACTIVITY", 
                                       datediff(lit(ultima_fecha), col("DATE")))

print("✅ Datos preparados")

# %% [markdown]
# ## 📈 SECCIÓN 1: KPIs DE ACTIVIDAD DEL CLIENTE

# %% [code]
# ============================================================================
# CELDA 4: KPI 1 - ACTIVIDAD Y FRECUENCIA
# ============================================================================
print("📈 Calculando KPIs de actividad...")

# KPI 1.1: Frecuencia de transacciones por cliente
kpi_frecuencia = df_behaviour.groupBy("CLIENT_ID").agg(
    count("*").alias("TOTAL_TRANSACCIONES"),
    countDistinct("DATE").alias("DIAS_CON_ACTIVIDAD"),
    mean("CREDIT_CARD_DRAWINGS").alias("GASTO_PROMEDIO_DIARIO"),
    stddev("CREDIT_CARD_DRAWINGS").alias("GASTO_VARIABILIDAD")
)

# KPI 1.2: Recencia (RFM - Recency)
kpi_recencia = df_behaviour.groupBy("CLIENT_ID").agg(
    min("DAYS_SINCE_ACTIVITY").alias("DIAS_DESDE_ULTIMA_ACTIVIDAD")
)

# Unir KPIs de actividad
kpi_actividad = kpi_frecuencia.join(kpi_recencia, on="CLIENT_ID", how="inner")

# Segmentación por actividad
kpi_actividad = kpi_actividad.withColumn("SEGMENTO_ACTIVIDAD",
    when(col("DIAS_DESDE_ULTIMA_ACTIVIDAD") <= 30, "ACTIVO")
    .when(col("DIAS_DESDE_ULTIMA_ACTIVIDAD") <= 90, "MODERADO")
    .when(col("DIAS_DESDE_ULTIMA_ACTIVIDAD") <= 180, "POCO_ACTIVO")
    .otherwise("INACTIVO")
)

print("✅ KPIs de actividad calculados")

# %% [code]
# ============================================================================
# CELDA 5: VISUALIZACIÓN ACTIVIDAD
# ============================================================================
# Convertir a Pandas para visualización
kpi_actividad_pd = kpi_actividad.toPandas()

# Crear figura con subplots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Distribución de Segmentos de Actividad', 
                    'Transacciones por Segmento',
                    'Recencia vs Transacciones',
                    'Top 10 Clientes más Activos'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'bar'}]]
)

# Gráfico 1: Pie chart segmentos
segment_counts = kpi_actividad_pd['SEGMENTO_ACTIVIDAD'].value_counts()
fig.add_trace(
    go.Pie(labels=segment_counts.index, values=segment_counts.values, hole=0.3),
    row=1, col=1
)

# Gráfico 2: Transacciones promedio por segmento
segment_mean = kpi_actividad_pd.groupby('SEGMENTO_ACTIVIDAD')['TOTAL_TRANSACCIONES'].mean()
fig.add_trace(
    go.Bar(x=segment_mean.index, y=segment_mean.values, name='Transacciones Promedio'),
    row=1, col=2
)

# Gráfico 3: Dispersión recencia vs transacciones (muestra de 1000 clientes)
muestra = kpi_actividad_pd.sample(min(1000, len(kpi_actividad_pd)))
fig.add_trace(
    go.Scatter(x=muestra['DIAS_DESDE_ULTIMA_ACTIVIDAD'], 
               y=muestra['TOTAL_TRANSACCIONES'],
               mode='markers',
               marker=dict(size=8, opacity=0.6),
               name='Clientes'),
    row=2, col=1
)

# Gráfico 4: Top 10 clientes más activos
top_10 = kpi_actividad_pd.nlargest(10, 'TOTAL_TRANSACCIONES')
fig.add_trace(
    go.Bar(x=top_10['CLIENT_ID'].astype(str), y=top_10['TOTAL_TRANSACCIONES'],
           name='Top 10 Activos'),
    row=2, col=2
)

# Actualizar layout
fig.update_layout(height=800, showlegend=False, title_text="KPIs de Actividad del Cliente")
fig.show()

# %% [markdown]
# ## 💰 SECCIÓN 2: KPIs DE VALOR ECONÓMICO

# %% [code]
# ============================================================================
# CELDA 6: KPI 2 - VALOR ECONÓMICO Y RENTABILIDAD
# ============================================================================
print("💰 Calculando KPIs de valor económico...")

# KPI 2.1: Comportamiento financiero
kpi_financiero = df_behaviour.groupBy("CLIENT_ID").agg(
    sum("CREDIT_CARD_DRAWINGS").alias("GASTO_TOTAL"),
    avg("CREDIT_CARD_BALANCE").alias("SALDO_PROMEDIO"),
    max("CREDIT_CARD_LIMIT").alias("LIMITE_MAXIMO"),
    (sum("CREDIT_CARD_DRAWINGS") / count("*")).alias("GASTO_POR_TRANSACCION")
)

# Calcular uso del límite (%)
kpi_financiero = kpi_financiero.withColumn(
    "PORCENTAJE_USO_LIMITE",
    (col("SALDO_PROMEDIO") / col("LIMITE_MAXIMO")) * 100
)

# KPI 2.2: Información de clientes (ingresos, productos)
kpi_clientes = df_clients.select(
    "CLIENT_ID",
    "TOTAL_INCOME",
    "AMOUNT_PRODUCT",
    "NUMBER_OF_PRODUCTS",
    "NAME_PRODUCT_TYPE"
)

# Calcular rentabilidad estimada (ingresos - gastos)
kpi_valor = kpi_financiero.join(kpi_clientes, on="CLIENT_ID", how="inner")
kpi_valor = kpi_valor.withColumn(
    "RENTABILIDAD_ESTIMADA",
    col("TOTAL_INCOME") - col("GASTO_TOTAL")
)

# Segmentación por valor (ABC Analysis)
window_spec = Window.orderBy(col("GASTO_TOTAL").desc())
kpi_valor = kpi_valor.withColumn("RANK_VALOR", percent_rank().over(window_spec))

kpi_valor = kpi_valor.withColumn("SEGMENTO_VALOR",
    when(col("RANK_VALOR") <= 0.2, "A - ALTO VALOR")
    .when(col("RANK_VALOR") <= 0.5, "B - VALOR MEDIO")
    .otherwise("C - BAJO VALOR")
)

print("✅ KPIs de valor económico calculados")

# %% [code]
# ============================================================================
# CELDA 7: VISUALIZACIÓN VALOR ECONÓMICO
# ============================================================================
kpi_valor_pd = kpi_valor.toPandas()

# Crear dashboard de valor económico
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Distribución Segmentos de Valor', 
                    'Gasto Total por Segmento',
                    'Rentabilidad vs Ingresos',
                    'Uso del Límite de Crédito',
                    'Número de Productos por Cliente',
                    'Top 10 Clientes más Rentables'),
    specs=[[{'type': 'pie'}, {'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'histogram'}, {'type': 'box'}, {'type': 'bar'}]]
)

# 1. Distribución segmentos de valor
valor_counts = kpi_valor_pd['SEGMENTO_VALOR'].value_counts()
fig.add_trace(go.Pie(labels=valor_counts.index, values=valor_counts.values), row=1, col=1)

# 2. Gasto total por segmento
gasto_segmento = kpi_valor_pd.groupby('SEGMENTO_VALOR')['GASTO_TOTAL'].mean()
fig.add_trace(go.Bar(x=gasto_segmento.index, y=gasto_segmento.values), row=1, col=2)

# 3. Dispersión rentabilidad vs ingresos (muestra)
muestra_valor = kpi_valor_pd.sample(min(1000, len(kpi_valor_pd)))
fig.add_trace(
    go.Scatter(x=muestra_valor['TOTAL_INCOME'], 
               y=muestra_valor['RENTABILIDAD_ESTIMADA'],
               mode='markers',
               marker=dict(
                   size=8,
                   color=muestra_valor['PORCENTAJE_USO_LIMITE'],
                   colorscale='Viridis',
                   showscale=True,
                   colorbar=dict(title="% Uso Límite")
               )),
    row=1, col=3
)

# 4. Histograma uso del límite
fig.add_trace(
    go.Histogram(x=kpi_valor_pd['PORCENTAJE_USO_LIMITE'].clip(0, 100),
                 nbinsx=20,
                 name='Uso del Límite'),
    row=2, col=1
)

# 5. Boxplot productos por cliente
fig.add_trace(
    go.Box(y=kpi_valor_pd['NUMBER_OF_PRODUCTS'],
           name='Número de Productos'),
    row=2, col=2
)

# 6. Top 10 clientes más rentables
top_rentables = kpi_valor_pd.nlargest(10, 'RENTABILIDAD_ESTIMADA')
fig.add_trace(
    go.Bar(x=top_rentables['CLIENT_ID'].astype(str),
           y=top_rentables['RENTABILIDAD_ESTIMADA'],
           name='Top 10 Rentables'),
    row=2, col=3
)

fig.update_layout(height=900, title_text="KPIs de Valor Económico", showlegend=False)
fig.show()

# %% [markdown]
# ## 🎯 SECCIÓN 3: KPIs DE INTERACCIÓN Y FIDELIDAD

# %% [code]
# ============================================================================
# CELDA 8: KPI 3 - INTERACCIÓN Y FIDELIDAD
# ============================================================================
print("🎯 Calculando KPIs de interacción y fidelidad...")

# KPI 3.1: Interacción digital
kpi_digital = df_clients.select(
    "CLIENT_ID",
    "DIGITAL_CLIENT",
    "HOME_OWNER",
    "OWN_INSURANCE_CAR"
).withColumn(
    "CLIENTE_DIGITAL",
    when(col("DIGITAL_CLIENT") == 1, "SI").otherwise("NO")
)

# KPI 3.2: Comportamiento de fidelidad (retención)
# Calcular meses consecutivos con actividad
window_cliente = Window.partitionBy("CLIENT_ID").orderBy("YEAR", "MONTH")
kpi_fidelidad = df_behaviour.groupBy("CLIENT_ID", "YEAR", "MONTH").agg(
    count("*").alias("TRANSACCIONES_MES")
).withColumn(
    "MES_ANTERIOR",
    lag("MONTH", 1).over(window_cliente)
).withColumn(
    "ES_CONSECUTIVO",
    when(col("MONTH") - col("MES_ANTERIOR") == 1, 1).otherwise(0)
)

# Calcular racha máxima de meses consecutivos
kpi_racha = kpi_fidelidad.groupBy("CLIENT_ID").agg(
    sum("ES_CONSECUTIVO").alias("MESES_CONSECUTIVOS"),
    count("*").alias("MESES_TOTALES")
).withColumn(
    "TASA_RETENCION",
    col("MESES_CONSECUTIVOS") / col("MESES_TOTALES") * 100
)

# Unir todos los KPIs de fidelidad
kpi_fidelidad_final = kpi_digital.join(kpi_racha, on="CLIENT_ID", how="left")

# Segmentación por fidelidad
kpi_fidelidad_final = kpi_fidelidad_final.withColumn("SEGMENTO_FIDELIDAD",
    when(col("TASA_RETENCION") >= 80, "ALTA_FIDELIDAD")
    .when(col("TASA_RETENCION") >= 50, "MEDIA_FIDELIDAD")
    .otherwise("BAJA_FIDELIDAD")
).fillna({"TASA_RETENCION": 0, "SEGMENTO_FIDELIDAD": "SIN_HISTORIAL"})

print("✅ KPIs de fidelidad calculados")

# %% [code]
# ============================================================================
# CELDA 9: VISUALIZACIÓN FIDELIDAD
# ============================================================================
kpi_fidelidad_pd = kpi_fidelidad_final.toPandas()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Clientes Digitales vs No Digitales',
                    'Distribución Tasa de Retención',
                    'Fidelidad por Segmento Digital',
                    'Relación Propiedad vs Fidelidad'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

# 1. Clientes digitales
digital_counts = kpi_fidelidad_pd['CLIENTE_DIGITAL'].value_counts()
fig.add_trace(go.Pie(labels=digital_counts.index, values=digital_counts.values), row=1, col=1)

# 2. Histograma tasa de retención
fig.add_trace(
    go.Histogram(x=kpi_fidelidad_pd['TASA_RETENCION'].clip(0, 100),
                 nbinsx=20,
                 name='Tasa Retención'),
    row=1, col=2
)

# 3. Fidelidad por segmento digital
fidelidad_digital = kpi_fidelidad_pd.groupby(['CLIENTE_DIGITAL', 'SEGMENTO_FIDELIDAD']).size().unstack()
for segmento in fidelidad_digital.columns:
    fig.add_trace(
        go.Bar(x=fidelidad_digital.index, y=fidelidad_digital[segmento], name=segmento),
        row=2, col=1
    )

# 4. Dispersión propiedad vs fidelidad
fig.add_trace(
    go.Scatter(x=kpi_fidelidad_pd['HOME_OWNER'],
               y=kpi_fidelidad_pd['TASA_RETENCION'],
               mode='markers',
               marker=dict(
                   size=10,
                   color=kpi_fidelidad_pd['OWN_INSURANCE_CAR'],
                   colorscale='RdYlGn',
                   showscale=True,
                   colorbar=dict(title="Seguro Auto")
               ),
               text=kpi_fidelidad_pd['CLIENTE_DIGITAL']),
    row=2, col=2
)

fig.update_layout(height=700, title_text="KPIs de Interacción y Fidelidad", barmode='stack')
fig.show()

# %% [markdown]
# ## ⚠️ SECCIÓN 4: KPIs DE RIESGO POTENCIAL

# %% [code]
# ============================================================================
# CELDA 10: KPI 4 - RIESGO POTENCIAL
# ============================================================================
print("⚠️ Calculando KPIs de riesgo...")

# KPI 4.1: Riesgo financiero
kpi_riesgo = df_clients.select(
    "CLIENT_ID",
    "REACTIVE_SCORING",
    "PROACTIVE_SCORING", 
    "BEHAVIORAL_SCORING",
    "NON_COMPLIANT_CONTRACT",
    "INSTALLMENT",
    "TOTAL_INCOME"
)

# Calcular score promedio y ratios de riesgo
kpi_riesgo = kpi_riesfo.withColumn(
    "SCORE_PROMEDIO",
    (col("REACTIVE_SCORING") + col("PROACTIVE_SCORING") + col("BEHAVIORAL_SCORING")) / 3
).withColumn(
    "RATIO_DEUDA_INGRESO",
    when(col("TOTAL_INCOME") > 0, col("INSTALLMENT") / col("TOTAL_INCOME") * 100)
    .otherwise(100)
)

# KPI 4.2: Riesgo de abandono (churn)
kpi_churn = kpi_actividad.select(
    "CLIENT_ID",
    "DIAS_DESDE_ULTIMA_ACTIVIDAD",
    "SEGMENTO_ACTIVIDAD"
).join(
    kpi_fidelidad_final.select("CLIENT_ID", "TASA_RETENCION"),
    on="CLIENT_ID",
    how="inner"
)

# Calcular probabilidad de churn (simplificado)
kpi_churn = kpi_churn.withColumn("PROBABILIDAD_CHURN",
    when(col("DIAS_DESDE_ULTIMA_ACTIVIDAD") > 180, 0.9)
    .when(col("DIAS_DESDE_ULTIMA_ACTIVIDAD") > 90, 0.7)
    .when(col("DIAS_DESDE_ULTIMA_ACTIVIDAD") > 30, 0.4)
    .when(col("TASA_RETENCION") < 30, 0.6)
    .otherwise(0.2)
)

# Unir todos los KPIs de riesgo
kpi_riesgo_final = kpi_riesgo.join(kpi_churn, on="CLIENT_ID", how="inner")

# Segmentación por riesgo
kpi_riesgo_final = kpi_riesgo_final.withColumn("NIVEL_RIESGO",
    when(col("PROBABILIDAD_CHURN") >= 0.7, "ALTO_RIESGO")
    .when(col("PROBABILIDAD_CHURN") >= 0.4, "MEDIO_RIESGO")
    .when(col("NON_COMPLIANT_CONTRACT") == 1, "RIESGO_CONTRATO")
    .when(col("RATIO_DEUDA_INGRESO") > 40, "RIESGO_FINANCIERO")
    .otherwise("BAJO_RIESGO")
)

print("✅ KPIs de riesgo calculados")

# %% [code]
# ============================================================================
# CELDA 11: VISUALIZACIÓN RIESGO
# ============================================================================
kpi_riesgo_pd = kpi_riesgo_final.toPandas()

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Distribución Niveles de Riesgo',
                    'Probabilidad de Churn',
                    'Score Promedio vs Deuda/Ingreso',
                    'Riesgo por Segmento de Actividad',
                    'Clientes con Contrato No Cumplido',
                    'Top 10 Clientes de Alto Riesgo'),
    specs=[[{'type': 'pie'}, {'type': 'histogram'}, {'type': 'scatter'}],
           [{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)

# 1. Distribución niveles de riesgo
riesgo_counts = kpi_riesgo_pd['NIVEL_RIESGO'].value_counts()
fig.add_trace(go.Pie(labels=riesgo_counts.index, values=riesgo_counts.values), row=1, col=1)

# 2. Histograma probabilidad de churn
fig.add_trace(
    go.Histogram(x=kpi_riesgo_pd['PROBABILIDAD_CHURN'],
                 nbinsx=20,
                 name='Probabilidad Churn'),
    row=1, col=2
)

# 3. Dispersión score vs ratio deuda/ingreso
muestra_riesgo = kpi_riesgo_pd.sample(min(1000, len(kpi_riesgo_pd)))
fig.add_trace(
    go.Scatter(x=muestra_riesgo['SCORE_PROMEDIO'],
               y=muestra_riesgo['RATIO_DEUDA_INGRESO'],
               mode='markers',
               marker=dict(
                   size=10,
                   color=muestra_riesgo['PROBABILIDAD_CHURN'],
                   colorscale='RdYlGn_r',
                   showscale=True,
                   colorbar=dict(title="Prob. Churn")
               ),
               text=muestra_riesgo['NIVEL_RIESGO']),
    row=1, col=3
)

# 4. Riesgo por segmento de actividad
riesgo_actividad = kpi_riesgo_pd.groupby('SEGMENTO_ACTIVIDAD')['PROBABILIDAD_CHURN'].mean()
fig.add_trace(go.Bar(x=riesgo_actividad.index, y=riesgo_actividad.values), row=2, col=1)

# 5. Clientes con contrato no cumplido
contrato_no_cumplido = kpi_riesgo_pd[kpi_riesgo_pd['NON_COMPLIANT_CONTRACT'] == 1]
if len(contrato_no_cumplido) > 0:
    fig.add_trace(
        go.Bar(x=['Contrato No Cumplido'], 
               y=[len(contrato_no_cumplido)],
               name='Contrato No Cumplido'),
        row=2, col=2
    )

# 6. Top 10 clientes de alto riesgo
alto_riesgo = kpi_riesgo_pd[kpi_riesgo_pd['NIVEL_RIESGO'] == 'ALTO_RIESGO']
if len(alto_riesgo) > 0:
    top_10_riesgo = alto_riesgo.nlargest(10, 'PROBABILIDAD_CHURN')
    fig.add_trace(
        go.Bar(x=top_10_riesgo['CLIENT_ID'].astype(str),
               y=top_10_riesgo['PROBABILIDAD_CHURN'],
               name='Alto Riesgo'),
        row=2, col=3
    )

fig.update_layout(height=800, title_text="KPIs de Riesgo Potencial", showlegend=False)
fig.show()

# %% [markdown]
# ## 🚀 SECCIÓN 5: KPIs DE OPORTUNIDADES COMERCIALES

# %% [code]
# ============================================================================
# CELDA 12: KPI 5 - OPORTUNIDADES COMERCIALES
# ============================================================================
print("🚀 Calculando KPIs de oportunidades...")

# Unir todos los KPIs para análisis cruzado
kpi_completo = kpi_actividad \
    .join(kpi_valor, on="CLIENT_ID", how="inner") \
    .join(kpi_fidelidad_final, on="CLIENT_ID", how="inner") \
    .join(kpi_riesgo_final, on="CLIENT_ID", how="inner")

# KPI 5.1: Oportunidades de cross-selling
kpi_oportunidades = kpi_completo.withColumn("OPORTUNIDAD_CROSS_SELLING",
    when((col("NUMBER_OF_PRODUCTS") == 1) & 
         (col("SEGMENTO_VALOR").isin(["A - ALTO VALOR", "B - VALOR MEDIO"])), 
         "ALTA_OPORTUNIDAD")
    .when((col("NUMBER_OF_PRODUCTS") <= 2) & 
          (col("TOTAL_INCOME") > col("TOTAL_INCOME").mean()), 
          "MEDIA_OPORTUNIDAD")
    .otherwise("BAJA_OPORTUNIDAD")
)

# KPI 5.2: Oportunidades de engagement (re-engagement)
kpi_oportunidades = kpi_oportunidades.withColumn("OPORTUNIDAD_ENGAGEMENT",
    when((col("SEGMENTO_ACTIVIDAD") == "INACTIVO") & 
         (col("SEGMENTO_VALOR") == "A - ALTO VALOR"), 
         "RECUPERACION_ALTO_VALOR")
    .when((col("SEGMENTO_ACTIVIDAD").isin(["POCO_ACTIVO", "INACTIVO"])) & 
          (col("CLIENTE_DIGITAL") == "NO"), 
          "MIGRACION_DIGITAL")
    .when((col("OWN_INSURANCE_CAR") == 0) & 
          (col("CAR_AGE").isNotNull()), 
          "SEGURO_AUTOMOVIL")
    .otherwise("MANTENIMIENTO")
)

# KPI 5.3: Clientes ideales para up-selling
kpi_oportunidades = kpi_oportunidades.withColumn("OPORTUNIDAD_UP_SELLING",
    when((col("PORCENTAJE_USO_LIMITE") > 70) & 
         (col("SCORE_PROMEDIO") > 70), 
         "AUMENTO_LIMITE_CREDITO")
    .when((col("RENTABILIDAD_ESTIMADA") > 0) & 
          (col("NUMBER_OF_PRODUCTS") < 3), 
          "PRODUCTOS_PREMIUM")
    .otherwise("NO_APLICA")
)

print("✅ KPIs de oportunidades calculados")

# %% [code]
# ============================================================================
# CELDA 13: VISUALIZACIÓN OPORTUNIDADES
# ============================================================================
kpi_oportunidades_pd = kpi_oportunidades.toPandas()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Oportunidades de Cross-Selling',
                    'Oportunidades de Engagement',
                    'Segmentación por Valor y Actividad',
                    'Clientes Ideales para Up-Selling'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'sunburst'}]]
)

# 1. Oportunidades cross-selling
cross_selling_counts = kpi_oportunidades_pd['OPORTUNIDAD_CROSS_SELLING'].value_counts()
fig.add_trace(
    go.Bar(x=cross_selling_counts.index, y=cross_selling_counts.values,
           name='Cross-Selling'),
    row=1, col=1
)

# 2. Oportunidades engagement
engagement_counts = kpi_oportunidades_pd['OPORTUNIDAD_ENGAGEMENT'].value_counts()
fig.add_trace(
    go.Bar(x=engagement_counts.index, y=engagement_counts.values,
           name='Engagement'),
    row=1, col=2
)

# 3. Dispersión valor vs actividad (para identificar oportunidades)
fig.add_trace(
    go.Scatter(x=kpi_oportunidades_pd['TOTAL_TRANSACCIONES'],
               y=kpi_oportunidades_pd['GASTO_TOTAL'],
               mode='markers',
               marker=dict(
                   size=10,
                   color=kpi_oportunidades_pd['SEGMENTO_ACTIVIDAD'].map({
                       'ACTIVO': 'green',
                       'MODERADO': 'yellow',
                       'POCO_ACTIVO': 'orange',
                       'INACTIVO': 'red'
                   }),
                   opacity=0.7
               ),
               text=kpi_oportunidades_pd['OPORTUNIDAD_CROSS_SELLING'],
               hoverinfo='text+x+y'),
    row=2, col=1
)

# 4. Sunburst: Jerarquía de oportunidades
kpi_oportunidades_pd['COUNT'] = 1
sunburst_data = kpi_oportunidades_pd.groupby(
    ['SEGMENTO_VALOR', 'SEGMENTO_ACTIVIDAD', 'OPORTUNIDAD_CROSS_SELLING']
)['COUNT'].sum().reset_index()

fig.add_trace(
    go.Sunburst(
        labels=sunburst_data['SEGMENTO_VALOR'] + ' - ' + 
               sunburst_data['SEGMENTO_ACTIVIDAD'] + ' - ' +
               sunburst_data['OPORTUNIDAD_CROSS_SELLING'],
        parents=['' if pd.isna(x) else 
                f"{sunburst_data.loc[i, 'SEGMENTO_VALOR']} - {sunburst_data.loc[i, 'SEGMENTO_ACTIVIDAD']}" 
                for i, x in enumerate(sunburst_data['OPORTUNIDAD_CROSS_SELLING'])],
        values=sunburst_data['COUNT'],
        branchvalues="total",
        hoverinfo="label+value+percent parent"
    ),
    row=2, col=2
)

fig.update_layout(height=800, title_text="KPIs de Oportunidades Comerciales")
fig.show()

# %% [markdown]
# ## 📋 SECCIÓN 6: RESUMEN EJECUTIVO Y EXPORTACIÓN

# %% [code]
# ============================================================================
# CELDA 14: RESUMEN EJECUTIVO DE KPIs
# ============================================================================
print("📋 Generando resumen ejecutivo...")

# Calcular métricas resumen
resumen_kpis = {
    "TOTAL_CLIENTES": kpi_completo.count(),
    "CLIENTES_ACTIVOS": kpi_completo.filter(col("SEGMENTO_ACTIVIDAD") == "ACTIVO").count(),
    "CLIENTES_DIGITALES": kpi_completo.filter(col("DIGITAL_CLIENT") == 1).count(),
    "CLIENTES_ALTO_VALOR": kpi_completo.filter(col("SEGMENTO_VALOR") == "A - ALTO VALOR").count(),
    "CLIENTES_ALTO_RIESGO": kpi_completo.filter(col("NIVEL_RIESGO") == "ALTO_RIESGO").count(),
    "TASA_ACTIVIDAD_PROMEDIO": kpi_completo.agg(avg("TOTAL_TRANSACCIONES")).collect()[0][0],
    "GASTO_TOTAL_PROMEDIO": kpi_completo.agg(avg("GASTO_TOTAL")).collect()[0][0],
    "RENTABILIDAD_PROMEDIO": kpi_completo.agg(avg("RENTABILIDAD_ESTIMADA")).collect()[0][0],
    "TASA_RETENCION_PROMEDIO": kpi_completo.agg(avg("TASA_RETENCION")).collect()[0][0],
    "OPORTUNIDADES_CROSS_SELLING": kpi_completo.filter(
        col("OPORTUNIDAD_CROSS_SELLING") == "ALTA_OPORTUNIDAD"
    ).count()
}

# Mostrar resumen
print("\n" + "="*60)
print("📊 RESUMEN EJECUTIVO - FINPLUS ANALYTICS")
print("="*60)
for kpi, valor in resumen_kpis.items():
    if isinstance(valor, float):
        print(f"{kpi.replace('_', ' ').title():40} {valor:10.2f}")
    else:
        print(f"{kpi.replace('_', ' ').title():40} {valor:10,}")
print("="*60)

# %% [code]
# ============================================================================
# CELDA 15: EXPORTAR KPIs PARA DASHBOARD
# ============================================================================
print("💾 Exportando KPIs para dashboard...")

# Exportar KPIs agregados
kpi_completo.write.mode("overwrite").parquet("../data/curated/kpis_completos.parquet")

# Exportar resumen para Tableau/PowerBI
resumen_df = spark.createDataFrame([resumen_kpis])
resumen_df.write.mode("overwrite").csv("../data/curated